## 1.Dữ liệu

In [92]:
import numpy as np
import pandas as pd
from binance.client import Client
import warnings

warnings.filterwarnings("ignore")

# =========================
# 1. CẤU HÌNH ĐA TÀI SẢN VÀ HẰNG SỐ
# =========================
TARGET_SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT"]
PRED_HORIZON = 4
TRAIN_SPLIT = 0.85
USE_SENTIMENT_FEATURE = False 
LAG_STEPS_V8 = [1, 2, 3, 6, 12]

MIN_TP_PCT = 0.004
MAX_TP_PCT = 0.035
MIN_SL_PCT = 0.003
MAX_SL_PCT = 0.025

FEE_PER_SIDE = 0.001
ROUND_TRIP_FEE = FEE_PER_SIDE * 2
LABEL_NET_BUFFER = 0.0015
MIN_GROSS_PROFIT_FOR_POSITIVE = ROUND_TRIP_FEE + LABEL_NET_BUFFER

COIN_PARAMS = {
    "BTCUSDT": {"TP_ATR_MULT": 1.8, "SL_ATR_MULT": 1.2},
    "ETHUSDT": {"TP_ATR_MULT": 2.0, "SL_ATR_MULT": 1.4},
    "SOLUSDT": {"TP_ATR_MULT": 2.5, "SL_ATR_MULT": 1.5},
    "BNBUSDT": {"TP_ATR_MULT": 2.0, "SL_ATR_MULT": 1.3},
    "XRPUSDT": {"TP_ATR_MULT": 2.2, "SL_ATR_MULT": 1.4}
}

client = Client() 

# =========================
# 2. CÁC HÀM TIỀN XỬ LÝ
# =========================
def ensure_base_features_v8(df):
    df = df.copy()
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    if "Open time" in df.columns:
        if not pd.api.types.is_datetime64_any_dtype(df["Open time"]):
            sample = df["Open time"].dropna().iloc[0]
            if isinstance(sample, (int, float, np.integer, np.floating)):
                df["Open time"] = pd.to_datetime(df["Open time"], unit="ms", utc=True).dt.tz_localize(None)
            else:
                df["Open time"] = pd.to_datetime(df["Open time"], utc=True).dt.tz_localize(None)
    df["Return"] = df["Close"].pct_change()
    df["EMA_14"] = df["Close"].ewm(span=14, adjust=False).mean()
    df["EMA_50"] = df["Close"].ewm(span=50, adjust=False).mean()
    delta = df["Close"].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    df["RSI_14"] = 100 - (100 / (1 + (gain / loss.replace(0, np.nan))))
    df["MACD"] = df["Close"].ewm(span=12, adjust=False).mean() - df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]
    df["BB_Mid"] = df["Close"].rolling(window=20).mean()
    df["BB_Std"] = df["Close"].rolling(window=20).std()
    df["BB_Upper"] = df["BB_Mid"] + (df["BB_Std"] * 2)
    df["BB_Lower"] = df["BB_Mid"] - (df["BB_Std"] * 2)
    if "Open time" in df.columns:
        df["Hour"] = df["Open time"].dt.hour
        df["DayOfWeek"] = df["Open time"].dt.dayofweek
    else:
        df["Hour"], df["DayOfWeek"] = 0, 0
    return df

def engineer_model_features_v8(df):
    df = df.copy()
    if "Sentiment_Score" not in df.columns: df["Sentiment_Score"] = 0.0
    else: df["Sentiment_Score"] = df["Sentiment_Score"].fillna(0.0)
    df["Log_Return_1"] = np.log(df["Close"]).diff()
    df["Return_3"] = df["Close"].pct_change(3)
    df["Return_6"] = df["Close"].pct_change(6)
    df["Return_12"] = df["Close"].pct_change(12)
    df["Return_24"] = df["Close"].pct_change(24)
    df["Range_Pct"] = (df["High"] - df["Low"]) / df["Close"].replace(0, np.nan)
    df["Body_Pct"] = (df["Close"] - df["Open"]) / df["Open"].replace(0, np.nan)
    df["Volume_Change"] = df["Volume"].pct_change().replace([np.inf, -np.inf], np.nan)
    volume_mean_24 = df["Volume"].rolling(24).mean()
    volume_std_24 = df["Volume"].rolling(24).std().replace(0, np.nan)
    df["Volume_Z"] = (df["Volume"] - volume_mean_24) / volume_std_24
    df["Volume_Regime"] = df["Volume"] / df["Volume"].rolling(72).mean().replace(0, np.nan) - 1.0
    prev_close = df["Close"].shift(1)
    true_range = pd.concat([df["High"] - df["Low"], (df["High"] - prev_close).abs(), (df["Low"] - prev_close).abs()], axis=1).max(axis=1)
    df["ATR_14"] = true_range.rolling(14).mean()
    df["ATR_Pct"] = df["ATR_14"] / df["Close"].replace(0, np.nan)
    df["ATR_Regime"] = df["ATR_Pct"] / df["ATR_Pct"].rolling(72).mean().replace(0, np.nan)
    df["Volatility_24"] = df["Return"].rolling(24).std()
    df["Volatility_Regime"] = df["Volatility_24"] / df["Volatility_24"].rolling(72).mean().replace(0, np.nan)
    df["EMA_14_Dist"] = (df["Close"] - df["EMA_14"]) / df["EMA_14"].replace(0, np.nan)
    df["EMA_50_Dist"] = (df["Close"] - df["EMA_50"]) / df["EMA_50"].replace(0, np.nan)
    df["Trend_Strength"] = (df["EMA_14"] - df["EMA_50"]) / df["Close"].replace(0, np.nan)
    df["EMA_14_Slope_3"] = df["EMA_14"].pct_change(3)
    df["EMA_50_Slope_6"] = df["EMA_50"].pct_change(6)
    df["MACD_Gap"] = df["MACD"] - df["MACD_Signal"]
    df["RSI_14_Norm"] = df["RSI_14"] / 100.0
    df["BB_Width"] = (df["BB_Upper"] - df["BB_Lower"]) / df["BB_Mid"].replace(0, np.nan)
    df["Breakout_20"] = df["Close"] / df["High"].rolling(20).max().shift(1).replace(0, np.nan) - 1.0
    df["Breakdown_20"] = df["Close"] / df["Low"].rolling(20).min().shift(1).replace(0, np.nan) - 1.0
    return df.replace([np.inf, -np.inf], np.nan)

def add_tree_lags_v8(df, lag_steps=LAG_STEPS_V8):
    df = df.copy()
    lag_cols = ["Log_Return_1", "Return_3", "Return_6", "Return_12", "Return_24", "EMA_14_Dist", "EMA_50_Dist", "Trend_Strength", "RSI_14_Norm", "MACD_Gap", "ATR_Pct", "Volume_Z", "Volume_Regime", "Volatility_24", "Volatility_Regime", "Breakout_20", "Breakdown_20"]
    for col in lag_cols:
        if col in df.columns:
            for lag in lag_steps: df[f"{col}_lag_{lag}"] = df[col].shift(lag)
    return df

def create_setup_filter_dual_v8(df):
    df = df.copy()
    # LONG
    df["Setup_Trend_Primary_L"] = ((df["EMA_14"] > df["EMA_50"]) & (df["Trend_Strength"] > -0.002)).astype(int)
    df["Setup_MACD_OK_L"] = (df["MACD_Gap"] > -0.0005).astype(int)
    df["Setup_RSI_OK_L"] = ((df["RSI_14_Norm"] >= 0.44) & (df["RSI_14_Norm"] <= 0.74)).astype(int)
    df["Setup_ATR_OK"] = (df["ATR_Pct"] <= 0.03).astype(int) 
    df["Setup_Vol_OK"] = (df["Volatility_Regime"] <= 1.9).astype(int) 
    df["Setup_Volume_OK"] = (df["Volume_Regime"] > -0.40).astype(int) 
    df["Setup_Breakout_OK_L"] = (df["Breakout_20"] > -0.03).astype(int)
    df["Setup_Slope_OK_L"] = ((df["EMA_14_Slope_3"] > -0.002) & (df["EMA_50_Slope_6"] > -0.003)).astype(int)
    df["Setup_Long_Score"] = df[["Setup_Trend_Primary_L", "Setup_MACD_OK_L", "Setup_RSI_OK_L", "Setup_ATR_OK", "Setup_Vol_OK", "Setup_Volume_OK", "Setup_Breakout_OK_L", "Setup_Slope_OK_L"]].sum(axis=1)
    df["setup_long_candidate"] = ((df["Setup_Long_Score"] >= 5) & (df["Setup_Trend_Primary_L"] == 1) & (df["Setup_ATR_OK"] == 1))

    # SHORT
    df["Setup_Trend_Primary_S"] = ((df["EMA_14"] < df["EMA_50"]) & (df["Trend_Strength"] < 0.002)).astype(int)
    df["Setup_MACD_OK_S"] = (df["MACD_Gap"] < 0.0005).astype(int)
    df["Setup_RSI_OK_S"] = ((df["RSI_14_Norm"] <= 0.56) & (df["RSI_14_Norm"] >= 0.26)).astype(int)
    df["Setup_Breakdown_OK_S"] = (df["Breakdown_20"] < 0.03).astype(int)
    df["Setup_Slope_OK_S"] = ((df["EMA_14_Slope_3"] < 0.002) & (df["EMA_50_Slope_6"] < 0.003)).astype(int)
    df["Setup_Short_Score"] = df[["Setup_Trend_Primary_S", "Setup_MACD_OK_S", "Setup_RSI_OK_S", "Setup_ATR_OK", "Setup_Vol_OK", "Setup_Volume_OK", "Setup_Breakdown_OK_S", "Setup_Slope_OK_S"]].sum(axis=1)
    df["setup_short_candidate"] = ((df["Setup_Short_Score"] >= 5) & (df["Setup_Trend_Primary_S"] == 1) & (df["Setup_ATR_OK"] == 1))
    return df

def create_dual_fee_aware_labels_v8(df, symbol="BTCUSDT", horizon=PRED_HORIZON):
    df = df.copy()
    labels_long = np.full(len(df), np.nan)
    realized_net_long = np.full(len(df), np.nan)
    exit_bars_long = np.full(len(df), np.nan)

    labels_short = np.full(len(df), np.nan)
    realized_net_short = np.full(len(df), np.nan)
    exit_bars_short = np.full(len(df), np.nan)

    closes, highs, lows, atr_pcts = df["Close"].values, df["High"].values, df["Low"].values, df["ATR_Pct"].values
    params = COIN_PARAMS.get(symbol, COIN_PARAMS["BTCUSDT"])
    tp_mult, sl_mult = params["TP_ATR_MULT"], params["SL_ATR_MULT"]

    for i in range(len(df) - horizon):
        entry, atr_pct = closes[i], atr_pcts[i]
        if not np.isfinite(entry) or not np.isfinite(atr_pct): continue

        tp_pct = float(np.clip(atr_pct * tp_mult, MIN_TP_PCT, MAX_TP_PCT))
        sl_pct = float(np.clip(atr_pct * sl_mult, MIN_SL_PCT, MAX_SL_PCT))

        upper_l, lower_l = entry * (1.0 + tp_pct), entry * (1.0 - sl_pct)
        lower_s, upper_s = entry * (1.0 - tp_pct), entry * (1.0 + sl_pct)

        hit_l, hit_s = False, False
        ret_l, ret_s = np.nan, np.nan
        exit_l, exit_s = horizon, horizon

        for j in range(1, horizon + 1):
            high_j, low_j = highs[i + j], lows[i + j]

            # Phe Long
            if not hit_l:
                if high_j >= upper_l and low_j <= lower_l: ret_l, hit_l, exit_l = -sl_pct, True, j 
                elif high_j >= upper_l: ret_l, hit_l, exit_l = tp_pct, True, j
                elif low_j <= lower_l: ret_l, hit_l, exit_l = -sl_pct, True, j

            # Phe Short
            if not hit_s:
                if high_j >= upper_s and low_j <= lower_s: ret_s, hit_s, exit_s = -sl_pct, True, j 
                elif low_j <= lower_s: ret_s, hit_s, exit_s = tp_pct, True, j
                elif high_j >= upper_s: ret_s, hit_s, exit_s = -sl_pct, True, j

            if hit_l and hit_s: break

        if not hit_l: ret_l = (closes[i + horizon] / entry) - 1.0
        if not hit_s: ret_s = 1.0 - (closes[i + horizon] / entry) 

        labels_long[i] = 1 if (ret_l - ROUND_TRIP_FEE) >= MIN_GROSS_PROFIT_FOR_POSITIVE else 0
        labels_short[i] = 1 if (ret_s - ROUND_TRIP_FEE) >= MIN_GROSS_PROFIT_FOR_POSITIVE else 0
        realized_net_long[i], realized_net_short[i] = (ret_l - ROUND_TRIP_FEE), (ret_s - ROUND_TRIP_FEE)
        exit_bars_long[i], exit_bars_short[i] = exit_l, exit_s

    df["target_long"], df["realized_net_return_long"], df["exit_bars_long"] = labels_long, realized_net_long, exit_bars_long
    df["target_short"], df["realized_net_return_short"], df["exit_bars_short"] = labels_short, realized_net_short, exit_bars_short
    return df

def get_and_process_coin_data(symbol):
    print(f"⏳ Đang tải và xử lý dữ liệu 1 năm qua cho {symbol}...")
    klines = client.get_historical_klines(symbol, Client.KLINE_INTERVAL_1HOUR, "1 year ago UTC")
    df = pd.DataFrame(klines, columns=["Open time", "Open", "High", "Low", "Close", "Volume", "Close time", "Quote Asset", "Trades", "Taker Buy Base", "Taker Buy Quote", "Ignore"])
    df = ensure_base_features_v8(df)
    df = engineer_model_features_v8(df)
    df = add_tree_lags_v8(df)
    df = create_setup_filter_dual_v8(df)
    df = create_dual_fee_aware_labels_v8(df, symbol=symbol, horizon=PRED_HORIZON)
    df = df.dropna().reset_index(drop=True)
    
    train_size = int(len(df) * TRAIN_SPLIT)
    df_train = df.iloc[:train_size].copy()
    df_test = df.iloc[train_size:].copy()
    print(f"✅ {symbol}: {len(df_train)} nến Train, {len(df_test)} nến Test.")
    return df_train, df_test

# =========================
# 3. VÒNG LẶP GỘP DỮ LIỆU
# =========================
all_train_frames = []
all_test_frames = []

for sym in TARGET_SYMBOLS:
    df_tr, df_te = get_and_process_coin_data(sym)
    all_train_frames.append(df_tr)
    all_test_frames.append(df_te)

mega_train_df = pd.concat(all_train_frames, ignore_index=True)
mega_test_df = pd.concat(all_test_frames, ignore_index=True)
mega_train_df = mega_train_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n" + "="*40)
print(f"🌟 ĐÃ GỘP THÀNH CÔNG SIÊU DỮ LIỆU ĐA TÀI SẢN!")
print(f"Tổng số nến Train: {len(mega_train_df):,}")
print(f"Tổng số nến Test: {len(mega_test_df):,}")
print("="*40)

⏳ Đang tải và xử lý dữ liệu 1 năm qua cho BTCUSDT...
✅ BTCUSDT: 7351 nến Train, 1298 nến Test.
⏳ Đang tải và xử lý dữ liệu 1 năm qua cho ETHUSDT...
✅ ETHUSDT: 7351 nến Train, 1298 nến Test.
⏳ Đang tải và xử lý dữ liệu 1 năm qua cho SOLUSDT...
✅ SOLUSDT: 7346 nến Train, 1297 nến Test.
⏳ Đang tải và xử lý dữ liệu 1 năm qua cho BNBUSDT...
✅ BNBUSDT: 7351 nến Train, 1298 nến Test.
⏳ Đang tải và xử lý dữ liệu 1 năm qua cho XRPUSDT...
✅ XRPUSDT: 7351 nến Train, 1298 nến Test.

🌟 ĐÃ GỘP THÀNH CÔNG SIÊU DỮ LIỆU ĐA TÀI SẢN!
Tổng số nến Train: 36,750
Tổng số nến Test: 6,489


## 2. Xây dựng kiến trúc và huấn luyện mô hình

### Mô hình XGBOOST SHORT+LONG VÀ BACKTEST

In [96]:
import joblib
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

PROBA_SCAN = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

def build_xgb_v8_model(y_fit):
    positives = int(np.sum(y_fit == 1))
    negatives = int(np.sum(y_fit == 0))
    scale_pos_weight = negatives / max(positives, 1)

    model = XGBClassifier(
        objective="binary:logistic", n_estimators=320, max_depth=4,
        learning_rate=0.03, subsample=0.90, colsample_bytree=0.90,
        min_child_weight=8, gamma=0.0, reg_lambda=2.0, reg_alpha=0.5,
        scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1,
        tree_method="hist", eval_metric="logloss",
    )
    return model

def backtest_v8_dual(df_setups, proba, threshold, direction="long"):
    local = df_setups.copy().reset_index(drop=True)
    local["buy_proba"] = np.asarray(proba).flatten()
    
    target_col = f"target_{direction}"
    net_ret_col = f"realized_net_return_{direction}"
    exit_bars_col = f"exit_bars_{direction}"

    equity = 1.0
    trades = []
    i = 0
    while i < len(local):
        row = local.iloc[i]
        if row["buy_proba"] >= threshold:
            net_ret = float(row[net_ret_col])
            equity *= (1.0 + net_ret)
            trades.append({
                "idx": i, "buy_proba": float(row["buy_proba"]),
                "actual_label": int(row[target_col]), "net_return": net_ret,
                "exit_bars": int(row[exit_bars_col]), "win": net_ret > 0,
            })
            i += max(1, int(row[exit_bars_col]))
        else:
            i += 1

    trades_df = pd.DataFrame(trades)
    if len(trades_df) == 0:
        return {"trades": 0, "win_rate": 0.0, "profit_factor": 0.0, "final_equity": 1.0, "max_drawdown": 0.0}

    gains = trades_df.loc[trades_df["net_return"] > 0, "net_return"].sum()
    losses = -trades_df.loc[trades_df["net_return"] < 0, "net_return"].sum()
    profit_factor = gains / losses if losses > 0 else np.inf

    equity, peak, max_dd = 1.0, 1.0, 0.0
    for r in trades_df["net_return"]:
        equity *= (1.0 + float(r))
        peak = max(peak, equity)
        max_dd = max(max_dd, (peak - equity) / peak if peak > 0 else 0.0)

    return {
        "trades": int(len(trades_df)),
        "win_rate": float(trades_df["win"].mean() * 100),
        "profit_factor": float(profit_factor if np.isfinite(profit_factor) else 999.0),
        "final_equity": float(equity),
        "max_drawdown": float(max_dd * 100),
    }

# =========================
# BẮT ĐẦU HUẤN LUYỆN
# =========================
feature_columns_v8 = [c for c in mega_train_df.columns if c not in [
    "Open time", "Close time", "Ignore", "target_long", "target_short", 
    "realized_net_return_long", "realized_net_return_short", "exit_bars_long", "exit_bars_short",
    "setup_long_candidate", "setup_short_candidate"
]]

# ================= 1. MÔ HÌNH LONG =================
print("\n" + "="*40)
print("📈 ĐANG HUẤN LUYỆN MÔ HÌNH LONG (MULTI-ASSET)...")
print("="*40)
train_setups_long = mega_train_df[mega_train_df["setup_long_candidate"]].copy()
test_setups_long = mega_test_df[mega_test_df["setup_long_candidate"]].copy()

X_train_long = train_setups_long[feature_columns_v8].astype(np.float32)
y_train_long = train_setups_long["target_long"].astype(int)
X_test_long = test_setups_long[feature_columns_v8].astype(np.float32)
y_test_long = test_setups_long["target_long"].astype(int)

model_long = build_xgb_v8_model(y_train_long)
model_long.fit(X_train_long, y_train_long, verbose=False)
proba_test_long = model_long.predict_proba(X_test_long)[:, 1]

print(f"Tổng số Setup Train: {len(train_setups_long)} | Test: {len(test_setups_long)}")
print("\n--- KẾT QUẢ BACKTEST LONG ---")
for thr in PROBA_SCAN:
    out = backtest_v8_dual(test_setups_long, proba_test_long, threshold=thr, direction="long")
    print(f"thr={thr:.2f} | trades={out['trades']} | win={out['win_rate']:.2f}% | pf={out['profit_factor']:.3f} | equity={out['final_equity']:.4f}x | mdd={out['max_drawdown']:.2f}%")

joblib.dump(model_long, "xgb_v8_long_fee_aware_multi.pkl")


# ================= 2. MÔ HÌNH SHORT =================
print("\n" + "="*40)
print("📉 ĐANG HUẤN LUYỆN MÔ HÌNH SHORT (MULTI-ASSET)...")
print("="*40)
train_setups_short = mega_train_df[mega_train_df["setup_short_candidate"]].copy()
test_setups_short = mega_test_df[mega_test_df["setup_short_candidate"]].copy()

X_train_short = train_setups_short[feature_columns_v8].astype(np.float32)
y_train_short = train_setups_short["target_short"].astype(int)
X_test_short = test_setups_short[feature_columns_v8].astype(np.float32)
y_test_short = test_setups_short["target_short"].astype(int)

model_short = build_xgb_v8_model(y_train_short)
model_short.fit(X_train_short, y_train_short, verbose=False)
proba_test_short = model_short.predict_proba(X_test_short)[:, 1]

print(f"Tổng số Setup Train: {len(train_setups_short)} | Test: {len(test_setups_short)}")
print("\n--- KẾT QUẢ BACKTEST SHORT ---")
for thr in PROBA_SCAN:
    out = backtest_v8_dual(test_setups_short, proba_test_short, threshold=thr, direction="short")
    print(f"thr={thr:.2f} | trades={out['trades']} | win={out['win_rate']:.2f}% | pf={out['profit_factor']:.3f} | equity={out['final_equity']:.4f}x | mdd={out['max_drawdown']:.2f}%")

joblib.dump(model_short, "xgb_v8_short_fee_aware_multi.pkl")

# Lưu Meta Data
joblib.dump(
    {
        "lag_steps": LAG_STEPS_V8,
        "feature_columns": feature_columns_v8,
        "use_sentiment_feature": USE_SENTIMENT_FEATURE
    },
    "xgb_v8_meta.pkl"
)

print("\n🎉 Đã lưu 2 mô hình Siêu Trí Tuệ thành công!")


📈 ĐANG HUẤN LUYỆN MÔ HÌNH LONG (MULTI-ASSET)...
Tổng số Setup Train: 16929 | Test: 2768

--- KẾT QUẢ BACKTEST LONG ---
thr=0.45 | trades=574 | win=35.89% | pf=0.580 | equity=0.2140x | mdd=78.80%
thr=0.50 | trades=399 | win=35.84% | pf=0.543 | equity=0.2821x | mdd=72.07%
thr=0.55 | trades=235 | win=38.30% | pf=0.653 | equity=0.5646x | mdd=46.20%
thr=0.60 | trades=112 | win=33.93% | pf=0.561 | equity=0.6741x | mdd=35.08%
thr=0.65 | trades=35 | win=40.00% | pf=0.836 | equity=0.9566x | mdd=9.96%
thr=0.70 | trades=7 | win=42.86% | pf=1.340 | equity=1.0122x | mdd=2.70%

📉 ĐANG HUẤN LUYỆN MÔ HÌNH SHORT (MULTI-ASSET)...
Tổng số Setup Train: 16402 | Test: 3054

--- KẾT QUẢ BACKTEST SHORT ---
thr=0.45 | trades=565 | win=36.81% | pf=0.532 | equity=0.1985x | mdd=80.39%
thr=0.50 | trades=402 | win=38.81% | pf=0.612 | equity=0.4006x | mdd=60.28%
thr=0.55 | trades=247 | win=40.08% | pf=0.587 | equity=0.5295x | mdd=47.72%
thr=0.60 | trades=139 | win=38.85% | pf=0.537 | equity=0.6411x | mdd=36.70%
thr

### (AUTO-TUNING) VỚI OPTUNA

In [99]:
import optuna
import numpy as np
import pandas as pd
import joblib
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# Tắt các cảnh báo log rác của Optuna cho gọn màn hình
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Giả định bạn đã chạy Cell gộp dữ liệu trước đó và đã có biến `mega_train_df` trong RAM
if "mega_train_df" not in globals():
    raise ValueError("Chưa có mega_train_df! Vui lòng chạy Cell gộp dữ liệu 5 đồng coin trước.")

print("="*50)
print("🧬 KHỞI ĐỘNG QUÁ TRÌNH TỰ TIẾN HÓA (AUTO-TUNING) VỚI OPTUNA")
print("="*50)

# Lấy danh sách cột đặc trưng
feature_columns_v8 = [c for c in mega_train_df.columns if c not in [
    "Open time", "Close time", "Ignore", "target_long", "target_short", 
    "realized_net_return_long", "realized_net_return_short", "exit_bars_long", "exit_bars_short",
    "setup_long_candidate", "setup_short_candidate"
]]

# ==========================================
# 1. HÀM MỤC TIÊU CHO PHE LONG (MUA)
# ==========================================
def objective_long(trial):
    # Chia tập dữ liệu Tạm thời (Chronological Split) để đánh giá trong lúc tìm kiếm
    df_long = mega_train_df[mega_train_df["setup_long_candidate"]].copy()
    train_size = int(len(df_long) * 0.8) # Lấy 80% cũ làm Train, 20% mới làm Validation
    
    train_data = df_long.iloc[:train_size]
    val_data = df_long.iloc[train_size:]
    
    X_tr = train_data[feature_columns_v8].astype(np.float32)
    y_tr = train_data["target_long"].astype(int)
    X_val = val_data[feature_columns_v8].astype(np.float32)
    y_val = val_data["target_long"].astype(int)
    
    # Cân bằng nhãn
    positives = int(np.sum(y_tr == 1))
    scale_pos = (len(y_tr) - positives) / max(positives, 1)
    
    # Không gian tìm kiếm Siêu tham số (Gen đột biến)
    param = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1,
        'scale_pos_weight': scale_pos,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 2, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 2.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 3.0)
    }
    
    # Khởi tạo và huấn luyện
    model = XGBClassifier(**param)
    model.fit(X_tr, y_tr, verbose=False)
    
    # Đánh giá độ thông minh bằng chỉ số ROC-AUC trên tập Validation
    preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)
    return auc

# ==========================================
# 2. HÀM MỤC TIÊU CHO PHE SHORT (BÁN KHỐNG)
# ==========================================
def objective_short(trial):
    df_short = mega_train_df[mega_train_df["setup_short_candidate"]].copy()
    train_size = int(len(df_short) * 0.8)
    
    train_data = df_short.iloc[:train_size]
    val_data = df_short.iloc[train_size:]
    
    X_tr = train_data[feature_columns_v8].astype(np.float32)
    y_tr = train_data["target_short"].astype(int)
    X_val = val_data[feature_columns_v8].astype(np.float32)
    y_val = val_data["target_short"].astype(int)
    
    scale_pos = (len(y_tr) - int(np.sum(y_tr == 1))) / max(int(np.sum(y_tr == 1)), 1)
    
    param = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1,
        'scale_pos_weight': scale_pos,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 2, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 2.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 3.0)
    }
    
    model = XGBClassifier(**param)
    model.fit(X_tr, y_tr, verbose=False)
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

# ==========================================
# 3. THỰC THI TÌM KIẾM & THAY NÃO (RETRAIN)
# ==========================================
N_TRIALS = 30 # Có thể tăng lên 50 hoặc 100 nếu bạn có thời gian rảnh (VPS chạy đêm)

print("\nĐang tìm kiếm Bộ Gen tối ưu cho MÔ HÌNH LONG...")
study_long = optuna.create_study(direction="maximize")
study_long.optimize(objective_long, n_trials=N_TRIALS)
best_params_long = study_long.best_params
print(f"🏆 Gen LONG xịn nhất đạt AUC: {study_long.best_value:.4f}")
print(f"Chi tiết Gen LONG: {best_params_long}")

print("\nĐang tìm kiếm Bộ Gen tối ưu cho MÔ HÌNH SHORT...")
study_short = optuna.create_study(direction="maximize")
study_short.optimize(objective_short, n_trials=N_TRIALS)
best_params_short = study_short.best_params
print(f"🏆 Gen SHORT xịn nhất đạt AUC: {study_short.best_value:.4f}")
print(f"Chi tiết Gen SHORT: {best_params_short}")

# ==========================================
# 4. HUẤN LUYỆN LẠI TOÀN BỘ VỚI GEN MỚI VÀ LƯU FILE
# ==========================================
print("\n🔄 Đang tiến hành Thay Não (Huấn luyện lại trên 100% dữ liệu)...")

# Thay não LONG
df_long = mega_train_df[mega_train_df["setup_long_candidate"]].copy()
X_all_long = df_long[feature_columns_v8].astype(np.float32)
y_all_long = df_long["target_long"].astype(int)
best_params_long['scale_pos_weight'] = (len(y_all_long) - np.sum(y_all_long == 1)) / max(np.sum(y_all_long == 1), 1)

final_model_long = XGBClassifier(**best_params_long, random_state=42, tree_method='hist', n_jobs=-1)
final_model_long.fit(X_all_long, y_all_long, verbose=False)
joblib.dump(final_model_long, "xgb_v8_long_fee_aware_multi.pkl")

# Thay não SHORT
df_short = mega_train_df[mega_train_df["setup_short_candidate"]].copy()
X_all_short = df_short[feature_columns_v8].astype(np.float32)
y_all_short = df_short["target_short"].astype(int)
best_params_short['scale_pos_weight'] = (len(y_all_short) - np.sum(y_all_short == 1)) / max(np.sum(y_all_short == 1), 1)

final_model_short = XGBClassifier(**best_params_short, random_state=42, tree_method='hist', n_jobs=-1)
final_model_short.fit(X_all_short, y_all_short, verbose=False)
joblib.dump(final_model_short, "xgb_v8_short_fee_aware_multi.pkl")

print("\n🎉 TIẾN HÓA THÀNH CÔNG! Đã lưu đè Siêu Trí Tuệ mới vào ổ cứng.")

🧬 KHỞI ĐỘNG QUÁ TRÌNH TỰ TIẾN HÓA (AUTO-TUNING) VỚI OPTUNA

Đang tìm kiếm Bộ Gen tối ưu cho MÔ HÌNH LONG...
🏆 Gen LONG xịn nhất đạt AUC: 0.7719
Chi tiết Gen LONG: {'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.0780749557702527, 'subsample': 0.8003975671384206, 'colsample_bytree': 0.7610878990266572, 'min_child_weight': 3, 'gamma': 0.7539982836894039, 'reg_alpha': 0.29031737983427164, 'reg_lambda': 2.452759156296797}

Đang tìm kiếm Bộ Gen tối ưu cho MÔ HÌNH SHORT...
🏆 Gen SHORT xịn nhất đạt AUC: 0.7972
Chi tiết Gen SHORT: {'n_estimators': 450, 'max_depth': 6, 'learning_rate': 0.07887065195963541, 'subsample': 0.8451538403309157, 'colsample_bytree': 0.9711697917359647, 'min_child_weight': 10, 'gamma': 0.8177290764574827, 'reg_alpha': 0.9517268467248406, 'reg_lambda': 2.4340056244301698}

🔄 Đang tiến hành Thay Não (Huấn luyện lại trên 100% dữ liệu)...

🎉 TIẾN HÓA THÀNH CÔNG! Đã lưu đè Siêu Trí Tuệ mới vào ổ cứng.


### HUẤN LUYỆN AUTOENCODER

In [106]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" # Tắt log rác của Tensorflow

print("="*50)
print("🛡️ ĐANG HUẤN LUYỆN MẠNG AUTOENCODER DÒ TÌM BẤT THƯỜNG")
print("="*50)

# 1. Chọn các đặc trưng phản ánh trạng thái "Sức khỏe" của thị trường
anomaly_features = [
    "Volume_Z", "Volume_Regime", "ATR_Regime", 
    "Volatility_Regime", "Range_Pct", "Body_Pct"
]

# Chuẩn bị dữ liệu
df_anomaly = mega_train_df[anomaly_features].replace([np.inf, -np.inf], np.nan).dropna()
X_anomaly = df_anomaly.values

# Chuẩn hóa dữ liệu (Autoencoder rất nhạy cảm với scale)
scaler_ae = StandardScaler()
X_scaled = scaler_ae.fit_transform(X_anomaly)

# 2. Xây dựng Kiến trúc Autoencoder (Hình cái đồng hồ cát)
input_dim = X_scaled.shape[1]

# Lớp Nén (Encoder)
input_layer = Input(shape=(input_dim,))
encoded = Dense(16, activation='relu')(input_layer)
encoded = Dense(8, activation='relu')(encoded) # Nút cổ chai (Bottleneck)

# Lớp Giải nén (Decoder)
decoded = Dense(16, activation='relu')(encoded)
output_layer = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

print(f"Bắt đầu học {len(X_scaled)} mẫu trạng thái bình thường...")

# 3. Huấn luyện mô hình
autoencoder.fit(
    X_scaled, X_scaled,
    epochs=20,
    batch_size=256,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)

# 4. Tính toán Ngưỡng Báo động (Threshold)
print("\nĐang đo lường Lỗi tái tạo (Reconstruction Error)...")
X_pred = autoencoder.predict(X_scaled, verbose=0)
mse = np.mean(np.power(X_scaled - X_pred, 2), axis=1)

# Đặt ngưỡng là Percentile thứ 99.5 (Nghĩa là chỉ 0.5% dữ liệu bất thường nhất lịch sử mới kích hoạt)
anomaly_threshold = np.percentile(mse, 99.5)
print(f"🚨 Ngưỡng kích hoạt Kill-Switch (MSE Threshold): {anomaly_threshold:.4f}")

# 5. Lưu mô hình và thông số
autoencoder.save("autoencoder_killswitch.keras")
joblib.dump(scaler_ae, "scaler_ae.pkl")
joblib.dump({"threshold": anomaly_threshold, "features": anomaly_features}, "ae_meta.pkl")

print("🎉 Đã rèn luyện xong Bản năng sinh tồn!")

🛡️ ĐANG HUẤN LUYỆN MẠNG AUTOENCODER DÒ TÌM BẤT THƯỜNG
Bắt đầu học 36750 mẫu trạng thái bình thường...
Epoch 1/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.7624 - val_loss: 0.4479
Epoch 2/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3365 - val_loss: 0.2195
Epoch 3/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1750 - val_loss: 0.1351
Epoch 4/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1148 - val_loss: 0.0919
Epoch 5/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0814 - val_loss: 0.0688
Epoch 6/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0625 - val_loss: 0.0541
Epoch 7/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0462 - val_loss: 0.0388
Epoch 8/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0317 - val_loss: 0.0290
Epoch 9/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0241 - val_loss: 0.0243
Epoch 10/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0205 - val_loss: 0.0213
Epoch 11/20
130/130 ━━━━━━━━━━━━━━━━━

### BOT

In [1]:
import os
import json
import time
import warnings
from pathlib import Path
from decimal import Decimal
import xgboost as xgb

import feedparser
import joblib
import numpy as np
import pandas as pd
import requests
import schedule
from binance.client import Client

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

def ghi_log(thong_bao):
    print(thong_bao)
    with open("nhat_ky_trade_hybrid.txt", "a", encoding="utf-8") as f:
        f.write(thong_bao + "\n")

schedule.clear()

# =========================
# 1. CẤU HÌNH API & THÔNG SỐ CỐT LÕI
# =========================
API_KEY = "wR3cb7wAkuNPPxdxnMjWtlgQzQZ0xOlDxHkOJcduxrPRGcUn3DsUxzGops4tm8zf"
API_SECRET = "Ncw5w4mR23l7ZMFOvhS4xY4TP1C33ZnzZiV3BiIvYG6EwMuQVJzP4m506PNJeZ1g"
client = Client(API_KEY.strip(), API_SECRET.strip(), testnet=True)

TARGET_SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT"]

# Nâng ngưỡng vào lệnh dựa trên kết quả Backtest thực tế
BUY_PROBA_THRESHOLD = 0.68 

# --- THÔNG SỐ QUẢN LÝ VỐN ĐỘNG ---
BASE_BUDGET_USDT = 20.0   # Vốn cơ bản (68% - 74%)
MID_BUDGET_USDT = 35.0    # Vốn khá (75% - 79%)
HIGH_BUDGET_USDT = 50.0   # Vốn Tất tay (>= 80%)

# Thông số chốt lời/cắt lỗ động theo Đa tài sản
COIN_PARAMS = {
    "BTCUSDT": {"STOP_LOSS_PCT": 0.010, "TAKE_PROFIT_PCT": 0.018, "TRAILING_PCT": 0.007},
    "ETHUSDT": {"STOP_LOSS_PCT": 0.015, "TAKE_PROFIT_PCT": 0.025, "TRAILING_PCT": 0.010},
    "SOLUSDT": {"STOP_LOSS_PCT": 0.020, "TAKE_PROFIT_PCT": 0.035, "TRAILING_PCT": 0.015},
    "BNBUSDT": {"STOP_LOSS_PCT": 0.012, "TAKE_PROFIT_PCT": 0.020, "TRAILING_PCT": 0.008},
    "XRPUSDT": {"STOP_LOSS_PCT": 0.015, "TAKE_PROFIT_PCT": 0.025, "TRAILING_PCT": 0.010}
}
# =========================
# 2. NẠP DUAL MODELS, META & BẢN NĂNG SINH TỒN
# =========================
print("Đang nạp Siêu Trí Tuệ & Bản năng Sinh tồn... (Multi-Asset Dual V8) & Meta data...")

model_long = joblib.load("xgb_v8_long_fee_aware_multi.pkl")
model_short = joblib.load("xgb_v8_short_fee_aware_multi.pkl")
meta_v8 = joblib.load("xgb_v8_meta.pkl")
feature_columns_v8 = list(meta_v8["feature_columns"])
LAG_STEPS_V8 = meta_v8.get("lag_steps", [1, 2, 3, 6, 12])

from tensorflow.keras.models import load_model 
ae_model = load_model("autoencoder_killswitch.keras")
ae_scaler = joblib.load("scaler_ae.pkl")
ae_meta = joblib.load("ae_meta.pkl")
ae_threshold = ae_meta["threshold"]
ae_features = ae_meta["features"]

# ===================================================
# HỆ THỐNG GIẢI THÍCH (DÙNG XGBOOST NATIVE NÊN KHÔNG BỊ LỖI DLL)
# ===================================================
FEATURE_DICT = {
    "Volume_Z": "Đột biến Khối lượng (Z-Score)",
    "Volume_Regime": "Dòng tiền Vĩ mô",
    "RSI_14_Norm": "Sức mạnh Tương đối (RSI)",
    "MACD_Gap": "Động lượng MACD",
    "Trend_Strength": "Sức mạnh Xu hướng (EMA)",
    "ATR_Pct": "Độ nén giá (ATR)",
    "Breakout_20": "Cấu trúc Phá vỡ (Breakout)",
    "Volatility_Regime": "Biến động Thị trường"
}

def generate_ai_insights_native(model, X_live_df):
    """Bóc tách não bộ bằng tính năng SHAP nội bộ của XGBoost"""
    # Lấy lõi tính toán của XGBoost
    booster = model.get_booster()
    
    # Chuyển dữ liệu sang định dạng DMatrix của XGBoost
    dmatrix = xgb.DMatrix(X_live_df)
    
    # Kích hoạt pred_contribs=True để XGBoost tự tính giá trị SHAP
    contributions = booster.predict(dmatrix, pred_contribs=True)[0]
    
    # Giá trị cuối cùng trong mảng là độ lệch gốc (bias), ta bỏ đi, chỉ lấy sức ảnh hưởng của đặc trưng
    shap_values = contributions[:-1]
    
    feature_impacts = list(zip(feature_columns_v8, shap_values))
    
    pos_impacts = sorted([x for x in feature_impacts if x[1] > 0], key=lambda x: x[1], reverse=True)
    neg_impacts = sorted([x for x in feature_impacts if x[1] < 0], key=lambda x: x[1])
    
    insights = "\n   🔍 AI Insights (Bóc tách quyết định):"
    
    insights += "\n   [+] Các yếu tố ủng hộ:"
    for feat, val in pos_impacts[:3]:
        name = FEATURE_DICT.get(feat, feat)
        insights += f"\n       • {name} đang ủng hộ tích cực."
        
    if neg_impacts:
        insights += "\n   [-] Rủi ro lưu ý:"
        feat_neg = neg_impacts[0][0]
        name_neg = FEATURE_DICT.get(feat_neg, feat_neg)
        insights += f"\n       • Chú ý: {name_neg} đang hơi bất lợi."
        
    return insights
# =========================
# 3. QUẢN LÝ TRẠNG THÁI (STATE)
# =========================
STATE_FILE = Path("bot_runtime_state_dual.json")

def _default_symbol_state():
    return {
        "position_side": "NONE", "entry_price": 0.0, "quantity": 0.0,
        "invested_usdt": 0.0, "peak_price": 0.0, "trough_price": 0.0, "opened_at": None,
    }

def _load_runtime_state():
    if STATE_FILE.exists():
        try: return json.loads(STATE_FILE.read_text(encoding="utf-8"))
        except: pass
    return {"symbols": {}}

def _save_runtime_state():
    STATE_FILE.write_text(json.dumps(runtime_state, ensure_ascii=False, indent=2), encoding="utf-8")

runtime_state = _load_runtime_state()
bot_memory = runtime_state.setdefault("symbols", {})

# =========================
# 4. ASSET-SPECIFIC FINBERT
# =========================
def _fallback_sentiment_score(titles):
    positive_words = {"bull", "bullish", "surge", "rally", "rise", "up", "breakout", "gain"}
    negative_words = {"bear", "bearish", "drop", "fall", "down", "crash", "hack", "lawsuit"}
    if not titles: return 0.0
    scores = []
    for title in titles:
        text = str(title).lower()
        pos_hits = sum(word in text for word in positive_words)
        neg_hits = sum(word in text for word in negative_words)
        if pos_hits == 0 and neg_hits == 0: scores.append(0.0)
        else: scores.append((pos_hits - neg_hits) / (pos_hits + neg_hits))
    return float(np.mean(scores)) if scores else 0.0

def get_asset_sentiment(symbol):
    rss_url = "https://cointelegraph.com/rss"
    api_url = "https://api-inference.huggingface.co/models/ProsusAI/finbert"
    coin_keywords = {"BTCUSDT": ["bitcoin", "btc"], "ETHUSDT": ["ethereum", "eth"], "SOLUSDT": ["solana", "sol"]}
    keywords = coin_keywords.get(symbol, [symbol.replace("USDT", "").lower()])

    try:
        feed = feedparser.parse(rss_url)
        asset_titles = [entry.title for entry in feed.entries[:20] if any(k in entry.title.lower() for k in keywords)]
        if not asset_titles: return 0.0 
        response = requests.post(api_url, json={"inputs": asset_titles[:5]}, timeout=10)
        if response.status_code == 200:
            results = response.json()
            scores = []
            for res in results:
                best = max(res, key=lambda x: x["score"]) if isinstance(res, list) else max(results, key=lambda x: x["score"])
                if best["label"] == "positive": scores.append(best["score"])
                elif best["label"] == "negative": scores.append(-best["score"])
                else: scores.append(0.0)
            return float(np.mean(scores))
        return _fallback_sentiment_score(asset_titles)
    except:
        return _fallback_sentiment_score(asset_titles)

def analyze_order_book(symbol, depth=20):
    """
    Quét Sổ lệnh (Level-2) để đo lường áp lực Mua/Bán theo thời gian thực.
    Trả về: OFI (Độ lệch dòng lệnh), Tường Bán (Kháng cự gần), Tường Mua (Hỗ trợ gần)
    """
    try:
        # Kéo 20 tầng giá gần nhất của Sổ lệnh
        order_book = client.get_order_book(symbol=symbol, limit=depth)
        
        bids = order_book['bids'] # Phe mua kê ở dưới (Giá, Khối lượng)
        asks = order_book['asks'] # Phe bán chặn ở trên (Giá, Khối lượng)
        
        # Tính tổng khối lượng (USDT) phe Mua đang kê chờ
        total_bid_vol = sum(float(price) * float(qty) for price, qty in bids)
        # Tính tổng khối lượng (USDT) phe Bán đang chặn
        total_ask_vol = sum(float(price) * float(qty) for price, qty in asks)
        
        # 1. TÍNH CHỈ SỐ OFI (Order Flow Imbalance)
        # Chạy từ -1.0 (Áp lực bán tuyệt đối) đến 1.0 (Áp lực mua tuyệt đối)
        if (total_bid_vol + total_ask_vol) == 0:
            ofi = 0.0
        else:
            ofi = (total_bid_vol - total_ask_vol) / (total_bid_vol + total_ask_vol)
            
        # 2. Tìm "Bức tường" (Wall) lớn nhất trong 20 tầng giá
        biggest_bid = max(bids, key=lambda x: float(x[0]) * float(x[1]))
        biggest_ask = max(asks, key=lambda x: float(x[0]) * float(x[1]))
        
        return ofi, float(biggest_bid[0]), float(biggest_ask[0])
        
    except Exception as e:
        ghi_log(f"⚠️ Lỗi quét Sổ lệnh {symbol}: {e}")
        return 0.0, 0.0, 0.0
# =========================
# 5. TIỀN XỬ LÝ DỮ LIỆU
# =========================
def ensure_base_features_v8(df):
    df = df.copy()
    for col in ["Open", "High", "Low", "Close", "Volume"]: df[col] = pd.to_numeric(df[col])
    df["Open time"] = pd.to_datetime(df["Open time"], unit="ms", utc=True).dt.tz_localize(None)
    df["Return"] = df["Close"].pct_change()
    df["EMA_14"] = df["Close"].ewm(span=14, adjust=False).mean()
    df["EMA_50"] = df["Close"].ewm(span=50, adjust=False).mean()
    delta = df["Close"].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    df["RSI_14"] = 100 - (100 / (1 + (gain / loss.replace(0, np.nan))))
    df["MACD"] = df["Close"].ewm(span=12, adjust=False).mean() - df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]
    df["BB_Mid"] = df["Close"].rolling(window=20).mean()
    df["BB_Std"] = df["Close"].rolling(window=20).std()
    df["BB_Upper"] = df["BB_Mid"] + (df["BB_Std"] * 2)
    df["BB_Lower"] = df["BB_Mid"] - (df["BB_Std"] * 2)
    df["Hour"] = df["Open time"].dt.hour
    df["DayOfWeek"] = df["Open time"].dt.dayofweek
    return df

def engineer_model_features_v8(df):
    df = df.copy()
    df["Sentiment_Score"] = df.get("Sentiment_Score", 0.0).fillna(0.0)
    df["Log_Return_1"] = np.log(df["Close"]).diff()
    df["Return_3"] = df["Close"].pct_change(3)
    df["Return_6"] = df["Close"].pct_change(6)
    df["Return_12"] = df["Close"].pct_change(12)
    df["Return_24"] = df["Close"].pct_change(24)
    df["Range_Pct"] = (df["High"] - df["Low"]) / df["Close"].replace(0, np.nan)
    df["Body_Pct"] = (df["Close"] - df["Open"]) / df["Open"].replace(0, np.nan)
    df["Volume_Change"] = df["Volume"].pct_change().replace([np.inf, -np.inf], np.nan)
    volume_mean_24 = df["Volume"].rolling(24).mean()
    volume_std_24 = df["Volume"].rolling(24).std().replace(0, np.nan)
    df["Volume_Z"] = (df["Volume"] - volume_mean_24) / volume_std_24
    df["Volume_Regime"] = df["Volume"] / df["Volume"].rolling(72).mean().replace(0, np.nan) - 1.0
    prev_close = df["Close"].shift(1)
    tr = pd.concat([df["High"] - df["Low"], (df["High"] - prev_close).abs(), (df["Low"] - prev_close).abs()], axis=1).max(axis=1)
    df["ATR_14"] = tr.rolling(14).mean()
    df["ATR_Pct"] = df["ATR_14"] / df["Close"].replace(0, np.nan)
    df["ATR_Regime"] = df["ATR_Pct"] / df["ATR_Pct"].rolling(72).mean().replace(0, np.nan)
    df["Volatility_24"] = df["Return"].rolling(24).std()
    df["Volatility_Regime"] = df["Volatility_24"] / df["Volatility_24"].rolling(72).mean().replace(0, np.nan)
    df["EMA_14_Dist"] = (df["Close"] - df["EMA_14"]) / df["EMA_14"].replace(0, np.nan)
    df["EMA_50_Dist"] = (df["Close"] - df["EMA_50"]) / df["EMA_50"].replace(0, np.nan)
    df["Trend_Strength"] = (df["EMA_14"] - df["EMA_50"]) / df["Close"].replace(0, np.nan)
    df["EMA_14_Slope_3"] = df["EMA_14"].pct_change(3)
    df["EMA_50_Slope_6"] = df["EMA_50"].pct_change(6)
    df["MACD_Gap"] = df["MACD"] - df["MACD_Signal"]
    df["RSI_14_Norm"] = df["RSI_14"] / 100.0
    df["BB_Width"] = (df["BB_Upper"] - df["BB_Lower"]) / df["BB_Mid"].replace(0, np.nan)
    df["Breakout_20"] = df["Close"] / df["High"].rolling(20).max().shift(1).replace(0, np.nan) - 1.0
    df["Breakdown_20"] = df["Close"] / df["Low"].rolling(20).min().shift(1).replace(0, np.nan) - 1.0
    return df.replace([np.inf, -np.inf], np.nan)

def add_tree_lags_v8(df, lag_steps):
    df = df.copy()
    lag_cols = ["Log_Return_1", "Return_3", "Return_6", "Return_12", "Return_24", "EMA_14_Dist", "EMA_50_Dist", "Trend_Strength", "RSI_14_Norm", "MACD_Gap", "ATR_Pct", "Volume_Z", "Volume_Regime", "Volatility_24", "Volatility_Regime", "Breakout_20", "Breakdown_20"]
    for col in lag_cols:
        if col in df.columns:
            for lag in lag_steps: df[f"{col}_lag_{lag}"] = df[col].shift(lag)
    return df

def create_setup_filter_dual_v8(df):
    df = df.copy()
    # LONG
    df["Setup_Trend_Primary_L"] = ((df["EMA_14"] > df["EMA_50"]) & (df["Trend_Strength"] > -0.002)).astype(int)
    df["Setup_MACD_OK_L"] = (df["MACD_Gap"] > -0.0005).astype(int)
    df["Setup_RSI_OK_L"] = ((df["RSI_14_Norm"] >= 0.44) & (df["RSI_14_Norm"] <= 0.74)).astype(int)
    df["Setup_Breakout_OK_L"] = (df["Breakout_20"] > -0.03).astype(int)
    df["Setup_Slope_OK_L"] = ((df["EMA_14_Slope_3"] > -0.002) & (df["EMA_50_Slope_6"] > -0.003)).astype(int)
    
    # SHORT
    df["Setup_Trend_Primary_S"] = ((df["EMA_14"] < df["EMA_50"]) & (df["Trend_Strength"] < 0.002)).astype(int)
    df["Setup_MACD_OK_S"] = (df["MACD_Gap"] < 0.0005).astype(int)
    df["Setup_RSI_OK_S"] = ((df["RSI_14_Norm"] <= 0.56) & (df["RSI_14_Norm"] >= 0.26)).astype(int)
    df["Setup_Breakdown_OK_S"] = (df["Breakdown_20"] < 0.03).astype(int)
    df["Setup_Slope_OK_S"] = ((df["EMA_14_Slope_3"] < 0.002) & (df["EMA_50_Slope_6"] < 0.003)).astype(int)

    # SHARED
    df["Setup_ATR_OK"] = (df["ATR_Pct"] <= 0.03).astype(int)
    df["Setup_Vol_OK"] = (df["Volatility_Regime"] <= 1.9).astype(int)
    df["Setup_Volume_OK"] = (df["Volume_Regime"] > -0.40).astype(int)

    df["Setup_Long_Score"] = df[["Setup_Trend_Primary_L", "Setup_MACD_OK_L", "Setup_RSI_OK_L", "Setup_ATR_OK", "Setup_Vol_OK", "Setup_Volume_OK", "Setup_Breakout_OK_L", "Setup_Slope_OK_L"]].sum(axis=1)
    df["setup_long_candidate"] = ((df["Setup_Long_Score"] >= 5) & (df["Setup_Trend_Primary_L"] == 1) & (df["Setup_ATR_OK"] == 1))

    df["Setup_Short_Score"] = df[["Setup_Trend_Primary_S", "Setup_MACD_OK_S", "Setup_RSI_OK_S", "Setup_ATR_OK", "Setup_Vol_OK", "Setup_Volume_OK", "Setup_Breakdown_OK_S", "Setup_Slope_OK_S"]].sum(axis=1)
    df["setup_short_candidate"] = ((df["Setup_Short_Score"] >= 5) & (df["Setup_Trend_Primary_S"] == 1) & (df["Setup_ATR_OK"] == 1))
    return df

def prepare_live_feature_frame_dual(df, sentiment_score):
    df = df.copy()
    df = ensure_base_features_v8(df)
    df["Sentiment_Score"] = sentiment_score
    df = engineer_model_features_v8(df)
    df = add_tree_lags_v8(df, lag_steps=LAG_STEPS_V8)
    df = create_setup_filter_dual_v8(df)
    df["Setup_ATR_OK"] = df.get("Setup_ATR_OK", 0)
    df["Setup_Vol_OK"] = df.get("Setup_Vol_OK", 0)
    df["Setup_Volume_OK"] = df.get("Setup_Volume_OK", 0)
    df["Setup_Long_Score"] = df.get("Setup_Long_Score", 0)
    df["Setup_Short_Score"] = df.get("Setup_Short_Score", 0)
    df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return df

# =========================
# 6. KELLY CRITERION
# =========================
def get_dynamic_budget(probability):
    if probability >= 0.80: return HIGH_BUDGET_USDT, "CAO (Tất tay)"
    elif probability >= 0.75: return MID_BUDGET_USDT, "TRUNG BÌNH (Khá tự tin)"
    else: return BASE_BUDGET_USDT, "CƠ BẢN (Thăm dò)"

# ===================================================
# META-AGENT: LỚP CHỈ HUY QUẢN LÝ DANH MỤC
# ===================================================

CORRELATION_THRESHOLD = 0.75 # Ngưỡng báo động: Tương quan > 75% coi như đi chung một xu hướng

def get_correlation_matrix(symbols, lookback_hours=48):
    """
    Kéo dữ liệu giá Close của các đồng coin trong 48 giờ qua và tính Ma trận Tương quan.
    """
    price_data = {}
    try:
        for sym in symbols:
            klines = client.get_klines(symbol=sym, interval=Client.KLINE_INTERVAL_1HOUR, limit=lookback_hours)
            closes = [float(k[4]) for k in klines]
            price_data[sym] = closes
        
        # Chuyển thành DataFrame và tính mức sinh lời (Returns)
        df_prices = pd.DataFrame(price_data)
        df_returns = df_prices.pct_change().dropna()
        
        # Tính toán Ma trận Tương quan (Pearson)
        corr_matrix = df_returns.corr()
        return corr_matrix
    except Exception as e:
        ghi_log(f"⚠️ Meta-Agent lỗi khi tính Ma trận Tương quan: {e}")
        return None

def meta_agent_portfolio_optimizer(intended_trades, corr_matrix):
    """
    Lọc và điều phối ngân sách cho các Ý định giao dịch (Intended Trades).
    intended_trades: list các dictionary chứa {'symbol', 'action', 'proba', 'budget'}
    """
    if not intended_trades:
        return []
        
    if len(intended_trades) == 1 or corr_matrix is None:
        # Nếu chỉ có 1 tín hiệu, hoặc lỗi ma trận, cho phép đi qua bình thường
        return intended_trades

    ghi_log(f"⚖️ [META-AGENT] Đang thẩm định {len(intended_trades)} tín hiệu trùng lặp...")
    approved_trades = []
    
    # Chia phe để xử lý: Các lệnh LONG xử lý riêng, SHORT xử lý riêng
    long_trades = sorted([t for t in intended_trades if t['action'] == "OPEN_LONG"], key=lambda x: x['proba'], reverse=True)
    short_trades = sorted([t for t in intended_trades if t['action'] == "OPEN_SHORT"], key=lambda x: x['proba'], reverse=True)

    def process_directional_trades(trades_list, direction_name):
        approved = []
        for current_trade in trades_list:
            sym_current = current_trade['symbol']
            is_conflicted = False
            
            # So sánh lệnh hiện tại với các lệnh ĐÃ ĐƯỢC DUYỆT trước đó trong cùng phe
            for approved_trade in approved:
                sym_approved = approved_trade['symbol']
                corr_value = corr_matrix.loc[sym_current, sym_approved]
                
                if corr_value > CORRELATION_THRESHOLD:
                    ghi_log(f"🛑 [META-AGENT] CẢNH BÁO: Tương quan {sym_current} và {sym_approved} quá cao ({corr_value*100:.1f}%).")
                    
                    # CẮT GIẢM RỦI RO (Hạ ngân sách của lệnh yếu hơn)
                    # Giảm ngân sách lệnh hiện tại đi một nửa (hoặc hủy bỏ nếu vốn quá nhỏ)
                    new_budget = current_trade['budget'] / 2.0
                    current_trade['budget'] = new_budget
                    ghi_log(f"✂️ [META-AGENT] Đã hạ ngân sách của {sym_current} xuống còn ${new_budget:.2f} để chia sẻ rủi ro với {sym_approved}.")
                    is_conflicted = True
                    break # Chỉ cần đụng 1 tài sản trùng lặp là bị giảm ngân sách
                    
            approved.append(current_trade)
        return approved

    # Xử lý duyệt lệnh
    if long_trades:
        ghi_log(f"🔍 [META-AGENT] Xử lý cụm lệnh LONG: {[t['symbol'] for t in long_trades]}")
        approved_trades.extend(process_directional_trades(long_trades, "LONG"))
        
    if short_trades:
        ghi_log(f"🔍 [META-AGENT] Xử lý cụm lệnh SHORT: {[t['symbol'] for t in short_trades]}")
        approved_trades.extend(process_directional_trades(short_trades, "SHORT"))

    return approved_trades
# ===================================================
# META-AGENT: LỚP CHỈ HUY QUẢN LÝ DANH MỤC
# ===================================================

CORRELATION_THRESHOLD = 0.75 # Ngưỡng báo động: Tương quan > 75% coi như đi chung một xu hướng

def get_correlation_matrix(symbols, lookback_hours=48):
    """
    Kéo dữ liệu giá Close của các đồng coin trong 48 giờ qua và tính Ma trận Tương quan.
    """
    price_data = {}
    try:
        for sym in symbols:
            klines = client.get_klines(symbol=sym, interval=Client.KLINE_INTERVAL_1HOUR, limit=lookback_hours)
            closes = [float(k[4]) for k in klines]
            price_data[sym] = closes
        
        df_prices = pd.DataFrame(price_data)
        df_returns = df_prices.pct_change().dropna()
        corr_matrix = df_returns.corr()
        return corr_matrix
    except Exception as e:
        ghi_log(f"⚠️ Meta-Agent lỗi khi tính Ma trận Tương quan: {e}")
        return None

def meta_agent_portfolio_optimizer(intended_trades, corr_matrix):
    """
    Lọc và điều phối ngân sách cho các Ý định giao dịch (Intended Trades) để tránh rủi ro tương quan.
    """
    if not intended_trades: return []
    if len(intended_trades) == 1 or corr_matrix is None: return intended_trades

    ghi_log(f"\n⚖️ [META-AGENT] Đang thẩm định {len(intended_trades)} tín hiệu tiềm năng...")
    approved_trades = []
    
    long_trades = sorted([t for t in intended_trades if t['action'] == "OPEN_LONG"], key=lambda x: x['proba'], reverse=True)
    short_trades = sorted([t for t in intended_trades if t['action'] == "OPEN_SHORT"], key=lambda x: x['proba'], reverse=True)

    def process_directional_trades(trades_list, direction_name):
        approved = []
        for current_trade in trades_list:
            sym_current = current_trade['symbol']
            
            for approved_trade in approved:
                sym_approved = approved_trade['symbol']
                corr_value = corr_matrix.loc[sym_current, sym_approved]
                
                if corr_value > CORRELATION_THRESHOLD:
                    ghi_log(f"🛑 [META-AGENT] CẢNH BÁO: Tương quan {sym_current} và {sym_approved} quá cao ({corr_value*100:.1f}%).")
                    
                    # CẮT GIẢM RỦI RO (Hạ ngân sách của lệnh yếu hơn)
                    new_budget = current_trade['budget'] / 2.0
                    current_trade['budget'] = new_budget
                    ghi_log(f"✂️ [META-AGENT] Đã hạ ngân sách của {sym_current} xuống còn ${new_budget:.2f} để chia sẻ rủi ro với {sym_approved}.")
                    break
                    
            approved.append(current_trade)
        return approved

    if long_trades: approved_trades.extend(process_directional_trades(long_trades, "LONG"))
    if short_trades: approved_trades.extend(process_directional_trades(short_trades, "SHORT"))

    return approved_trades

def check_black_swan(df_features, symbol):
    """
    Quét qua Autoencoder. Nếu Lỗi tái tạo (MSE) > Ngưỡng -> Có Thiên nga đen!
    """
    try:
        live_row = df_features.iloc[[-1]][ae_features]
        X_scaled = ae_scaler.transform(live_row)
        X_pred = ae_model.predict(X_scaled, verbose=0)
        
        mse = np.mean(np.power(X_scaled - X_pred, 2), axis=1)[0]
        
        if mse > ae_threshold:
            ghi_log(f"🆘 [BÁO ĐỘNG ĐỎ] Phát hiện biến động DỊ THƯỜNG tại {symbol}! MSE = {mse:.2f} (Ngưỡng {ae_threshold:.2f})")
            return True # Có bất thường
        return False # An toàn
    except Exception as e:
        ghi_log(f"⚠️ Lỗi Autoencoder: {e}")
        return False

# ===================================================
# 7. TRÁI TIM TÁC TỬ (ĐÃ NÂNG CẤP META-AGENT)
# ===================================================
def run_dual_bot():
    ghi_log(f"\n--- [{pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}] QUÉT THỊ TRƯỜNG ĐA TÀI SẢN ---")

    # PHA 1: TRINH SÁT & THU THẬP Ý ĐỊNH
    intended_trades = []

    for symbol in TARGET_SYMBOLS:
        ghi_log(f"\n🔎 Đang soi Chart & Tin tức: {symbol}")
        try:
            now_ts = pd.Timestamp.now(tz="UTC").tz_localize(None).floor("s")
            memory = bot_memory.setdefault(symbol, _default_symbol_state())
            
            asset_sentiment = get_asset_sentiment(symbol) 
            ghi_log(f"📰 Điểm Báo chí {symbol} (FinBERT): {asset_sentiment:.4f}")

            klines = client.get_klines(symbol=symbol, interval=Client.KLINE_INTERVAL_1HOUR, limit=260)
            df = pd.DataFrame(klines, columns=["Open time", "Open", "High", "Low", "Close", "Volume", "Close time", "Quote Asset", "Trades", "Taker Buy Base", "Taker Buy Quote", "Ignore"])
            current_price = float(df["Close"].iloc[-1])

            df_features = prepare_live_feature_frame_dual(df, asset_sentiment)
            if len(df_features) == 0: continue

            pos_side = memory["position_side"] # Lấy pos_side sớm để kiểm tra

            # ===================================================
            # BẢN NĂNG SINH TỒN (AUTOENCODER KILL-SWITCH)
            # ===================================================
            is_black_swan = check_black_swan(df_features, symbol)
            if is_black_swan:
                ghi_log(f"💥 [KILL-SWITCH KÍCH HOẠT] Hệ thống nhận diện cú sốc thanh khoản ở {symbol}!")
                
                # Nếu đang có lệnh -> ĐÓNG NGAY LẬP TỨC BẰNG MỌI GIÁ (Thoát hiểm)
                if pos_side == "LONG":
                    ghi_log(f"🚑 Đang xả khẩn cấp lệnh LONG {symbol} để tháo chạy...")
                    # Gọi API đóng lệnh Market ở đây
                    memory.update(_default_symbol_state())
                elif pos_side == "SHORT":
                    ghi_log(f"🚑 Đang cắt khẩn cấp lệnh SHORT {symbol} để tháo chạy...")
                    # Gọi API đóng lệnh Market ở đây
                    memory.update(_default_symbol_state())
                
                # Bỏ qua hoàn toàn việc tìm lệnh mới cho đồng coin này
                continue 
            # ===================================================

            # Nếu bình yên vô sự thì AI mới bắt đầu phân tích...
            live_row = df_features.iloc[[-1]]
            X_live = live_row[feature_columns_v8].astype(np.float32)

            proba_long = float(model_long.predict_proba(X_live)[0][1])
            proba_short = float(model_short.predict_proba(X_live)[0][1])
            is_setup_long = bool(live_row["setup_long_candidate"].iloc[0])
            is_setup_short = bool(live_row["setup_short_candidate"].iloc[0])

            ghi_log(f"🧠 AI LONG: Tín hiệu={is_setup_long} | Xác suất={proba_long*100:.2f}%")
            ghi_log(f"🧠 AI SHORT: Tín hiệu={is_setup_short} | Xác suất={proba_short*100:.2f}%")

            action = "HOLD"
            active_proba = 0.0
            pos_side = memory["position_side"]

            exit_rules = COIN_PARAMS.get(symbol, COIN_PARAMS["BTCUSDT"])
            sym_sl = exit_rules["STOP_LOSS_PCT"]
            sym_tp = exit_rules["TAKE_PROFIT_PCT"]
            sym_trail = exit_rules["TRAILING_PCT"]

            # --- EXIT LOGIC (Ưu tiên chốt lời/cắt lỗ ngay lập tức) ---
            if pos_side == "LONG":
                memory["peak_price"] = max(float(memory["peak_price"]), current_price)
                pnl_pct = (current_price - float(memory["entry_price"])) / float(memory["entry_price"])
                trail_stop = float(memory["peak_price"]) * (1.0 - sym_trail)

                if pnl_pct <= -sym_sl: action = "CLOSE_LONG_SL"
                elif pnl_pct >= sym_tp: action = "CLOSE_LONG_TP"
                elif current_price <= trail_stop and pnl_pct > 0.005: action = "CLOSE_LONG_TRAIL"
                
            elif pos_side == "SHORT":
                memory["trough_price"] = min(float(memory["trough_price"]), current_price) if memory["trough_price"] > 0 else current_price
                pnl_pct = (float(memory["entry_price"]) - current_price) / float(memory["entry_price"]) 
                trail_stop = float(memory["trough_price"]) * (1.0 + sym_trail)

                if pnl_pct <= -sym_sl: action = "CLOSE_SHORT_SL"
                elif pnl_pct >= sym_tp: action = "CLOSE_SHORT_TP"
                elif current_price >= trail_stop and pnl_pct > 0.005: action = "CLOSE_SHORT_TRAIL"

            # --- ENTRY LOGIC ---
            if pos_side == "NONE" and action == "HOLD":
                if is_setup_long and proba_long >= BUY_PROBA_THRESHOLD:
                    action = "OPEN_LONG"
                    active_proba = proba_long
                elif is_setup_short and proba_short >= BUY_PROBA_THRESHOLD:
                    action = "OPEN_SHORT"
                    active_proba = proba_short

            # 1. NLP SENTIMENT GATEKEEPER
            if action == "OPEN_LONG" and asset_sentiment <= -0.15:
                action = "HOLD"
                ghi_log(f"🛑 [NLP GATEKEEPER] Hủy lệnh LONG {symbol} vì FUD!")
            elif action == "OPEN_SHORT" and asset_sentiment >= 0.15:
                action = "HOLD"
                ghi_log(f"🛑 [NLP GATEKEEPER] Hủy lệnh SHORT {symbol} vì FOMO!")

            # 2. MICROSTRUCTURE GATEKEEPER (LEVEL-2)
            if action in ["OPEN_LONG", "OPEN_SHORT"]:
                ofi, biggest_bid, biggest_ask = analyze_order_book(symbol, depth=20)
                ghi_log(f"👁️ [MẮT THẦN LEVEL-2] OFI: {ofi:+.2f} | Trợ giá: ${biggest_bid:,.2f} | Kháng cự: ${biggest_ask:,.2f}")
                
                if action == "OPEN_LONG" and ofi <= -0.30:
                    action = "HOLD"
                    ghi_log(f"🛑 [LEVEL-2] Hủy LONG {symbol}! Phe Bán đè giá nặng (OFI = {ofi:.2f}).")
                elif action == "OPEN_SHORT" and ofi >= 0.30:
                    action = "HOLD"
                    ghi_log(f"🛑 [LEVEL-2] Hủy SHORT {symbol}! Phe Mua đỡ giá mạnh (OFI = {ofi:.2f}).")

            # --- THỰC THI EXIT HOẶC LƯU Ý ĐỊNH ENTRY ---
            if action in ["OPEN_LONG", "OPEN_SHORT"]:
                # Thay vì MUA/BÁN ngay, lưu vào danh sách trình Meta-Agent duyệt
                trade_budget, budget_label = get_dynamic_budget(active_proba)
                intended_trades.append({
                    'symbol': symbol, 'action': action, 'proba': active_proba,
                    'budget': trade_budget, 'budget_label': budget_label, 'current_price': current_price
                })
            elif action.startswith("CLOSE"):
                # Đóng lệnh thì làm luôn để chạy nạn
                exit_type = action.split("_")[-1]
                profit_usd = memory.get("invested_usdt", 0) * pnl_pct
                ghi_log(f"✅ ĐÓNG LỆNH {pos_side} ({exit_type}). Lãi/Lỗ: ${profit_usd:.2f} ({pnl_pct*100:.2f}%)")
                memory.update(_default_symbol_state())
            else:
                if pos_side != "NONE":
                    profit_usd_tmp = memory.get("invested_usdt", 0) * pnl_pct
                    ghi_log(f"👀 Đang gồng {pos_side} (Vốn: ${memory.get('invested_usdt',0)}). Lời/Lỗ: ${profit_usd_tmp:.2f} ({pnl_pct*100:.2f}%)")

            _save_runtime_state()

        except Exception as e:
            ghi_log(f"❌ Lỗi xử lý {symbol}: {e}")

    # =================================================================
    # PHA 2 & 3: META-AGENT XỬ LÝ MA TRẬN VÀ XUẤT KÍCH LỆNH ĐƯỢC DUYỆT
    # =================================================================
    if intended_trades:
        try:
            corr_matrix = get_correlation_matrix(TARGET_SYMBOLS)
            final_approved_trades = meta_agent_portfolio_optimizer(intended_trades, corr_matrix)
            
            ghi_log(f"\n⚡ [THỰC THI] Bắt đầu triển khai {len(final_approved_trades)} chiến dịch được duyệt.")
            for trade in final_approved_trades:
                sym = trade['symbol']
                act = trade['action']
                budget = trade['budget']
                price = trade['current_price']
                proba = trade['proba'] # Lấy xác suất ra
                now_ts = pd.Timestamp.now(tz="UTC").tz_localize(None).floor("s")
                
                memory = bot_memory[sym]
                
                # --- TẠO LỜI GIẢI THÍCH SHAP ---
                # Lấy lại dòng dữ liệu X_live của đồng coin này để phân tích
                klines_tmp = client.get_klines(symbol=sym, interval=Client.KLINE_INTERVAL_1HOUR, limit=260)
                df_tmp = pd.DataFrame(klines_tmp, columns=["Open time", "Open", "High", "Low", "Close", "Volume", "Close time", "Quote Asset", "Trades", "Taker Buy Base", "Taker Buy Quote", "Ignore"])
                sentiment_tmp = get_asset_sentiment(sym)
                df_feat_tmp = prepare_live_feature_frame_dual(df_tmp, sentiment_tmp)
                X_live_tmp = df_feat_tmp.iloc[[-1]][feature_columns_v8].astype(np.float32)
                
                # --- IN THÔNG BÁO CHUẨN VIP ---
                if act == "OPEN_LONG":
                    # Dùng hàm Native
                    insights_msg = generate_ai_insights_native(model_long, X_live_tmp)
                    ghi_log(f"\n🚀 [TÍN HIỆU VIP] MỞ LONG {sym}")
                    ghi_log(f"   💰 Giá vào: ${price:,.4f} | Tự tin: {proba*100:.1f}%")
                    ghi_log(insights_msg)
                    memory.update({"position_side": "LONG", "entry_price": price, "peak_price": price, "invested_usdt": budget, "opened_at": str(now_ts)})
                else:
                    # Dùng hàm Native
                    insights_msg = generate_ai_insights_native(model_short, X_live_tmp)
                    ghi_log(f"\n🩸 [TÍN HIỆU VIP] MỞ SHORT {sym}")
                    ghi_log(f"   💰 Giá vào: ${price:,.4f} | Tự tin: {proba*100:.1f}%")
                    ghi_log(insights_msg)
                    memory.update({"position_side": "SHORT", "entry_price": price, "trough_price": price, "invested_usdt": budget, "opened_at": str(now_ts)})
                
                _save_runtime_state()

        except Exception as e:
            ghi_log(f"❌ Lỗi xuất kích từ Meta-Agent: {e}")

print("\n🤖 SIÊU TÁC TỬ SẴN SÀNG (MULTI-ASSET + DUAL V8 + NLP GATEKEEPER)")
run_dual_bot()
schedule.every().hour.at(":00").do(run_dual_bot)

while True:
    schedule.run_pending()
    time.sleep(1)

Đang nạp Siêu Trí Tuệ & Bản năng Sinh tồn... (Multi-Asset Dual V8) & Meta data...

🤖 SIÊU TÁC TỬ SẴN SÀNG (MULTI-ASSET + DUAL V8 + NLP GATEKEEPER)

--- [2026-04-09 22:03:54] QUÉT THỊ TRƯỜNG ĐA TÀI SẢN ---

🔎 Đang soi Chart & Tin tức: BTCUSDT
📰 Điểm Báo chí BTCUSDT (FinBERT): -0.1667
🧠 AI LONG: Tín hiệu=False | Xác suất=1.06%
🧠 AI SHORT: Tín hiệu=False | Xác suất=28.23%

🔎 Đang soi Chart & Tin tức: ETHUSDT
📰 Điểm Báo chí ETHUSDT (FinBERT): 0.0000
🆘 [BÁO ĐỘNG ĐỎ] Phát hiện biến động DỊ THƯỜNG tại ETHUSDT! MSE = 0.11 (Ngưỡng 0.02)
💥 [KILL-SWITCH KÍCH HOẠT] Hệ thống nhận diện cú sốc thanh khoản ở ETHUSDT!

🔎 Đang soi Chart & Tin tức: SOLUSDT
📰 Điểm Báo chí SOLUSDT (FinBERT): 0.0000
🧠 AI LONG: Tín hiệu=False | Xác suất=10.73%
🧠 AI SHORT: Tín hiệu=True | Xác suất=28.50%

🔎 Đang soi Chart & Tin tức: BNBUSDT
📰 Điểm Báo chí BNBUSDT (FinBERT): 0.0000
🧠 AI LONG: Tín hiệu=False | Xác suất=4.19%
🧠 AI SHORT: Tín hiệu=True | Xác suất=36.64%

🔎 Đang soi Chart & Tin tức: XRPUSDT
📰 Điểm Báo chí XRPUSDT 

KeyboardInterrupt: 